In [ ]:
pip install langchain langchain-text-splitters bs4 requests

In [ ]:
!pip install -q langchain-google-vertexai

In [ ]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [ ]:
pip install -U "langchain[openai]"

In [ ]:
!pip install -q langchain-community

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")



In [ ]:
!pip install ragas

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="o3-mini",
)

In [8]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
!pip install -q faiss-cpu

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
!pip install -q pypdf

In [ ]:
docs = PyPDFLoader('cs173textbook.pdf').load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    add_start_index= True
)

chunks = text_splitter.split_documents(docs)

print(f"Split blog post into {len(chunks)} sub-documents.")

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [15]:
document_ids = vectorstore.add_documents(documents = docs)

In [16]:
import json

In [18]:
with open('/content/discretemathtestsetA.json', 'r') as file:
  dataset = json.load(file)

In [21]:
with open('/content/discretemathtestsetB.json', 'r') as file:
  final_dataset = json.load(file)

In [ ]:
print(final_dataset)

In [19]:
baseline_model = ChatOpenAI(
    model="o3-mini",
)

In [20]:
results = {
    "baseline": [],
    "rag": []
}

In [20]:
final_results = {
    "baseline": [],
    "rag": []
}

In [ ]:
#final testing using final_dataset (60 questions)
print("testing baseline model, no context...")
final_results['baseline'] = []
for i, test in enumerate(final_dataset):
  query = test["query"]
  prompt = f"""
          You are a discrete math tutor.

          Answer the following question clearly and show all reasoning steps.

          Question: {query}
          """
  response = baseline_model.invoke(prompt)
  final_results["baseline"].append({
        "query_index": i,
        "query": query,
        "expected_answer": test["correct_answer"],
        "baseline_response": response.content,
  })

In [ ]:
import pandas as pd
df_final_baseline = pd.DataFrame(final_results['baseline'])
df_final_baseline

In [27]:
df_final_baseline.to_csv('/content/baseline_final_results.csv', index=False)

In [ ]:
#final 60-question data set testing for RAG model
print("testing RAG model with context...")
final_results['rag'] = []
for i, test in enumerate(final_dataset):
  query = test["query"]
  docs = vectorstore.similarity_search(query, k=4)

  context = "\n\n".join(d.page_content for d in docs)
  retrieved_pages = [int(d.metadata.get("page_label",0)) for d in docs]

  prompt = f"""
You are an expert CS173 discrete mathematics tutor.

Answer using only the provided context.

Rules:
- Use precise mathematical language.
- Make sure to use the same notation as the context (e.g. $W_n$)
- Base all definitions and claims strictly on the context.
- If the context is insufficient, explicitly say so.
- Justify all claims clearly and logically.
- Follow proof style consistent with the context when applicable.

Problem-specific guidance:
- For state machine creation: identify required state memory, define transitions, then check determinism and check if your solution can be optimized by merging states.
- Proofs: follow standard discrete math proof structure (direct, contradiction, induction as appropriate).
- For counting and set-theoretic questions, carefully distinguish finite, countably infinite, and uncountable sets.

Formatting:
- Present structured objects (trees, automata, diagrams) in clean readable formats (tables, dictionaries, or ASCII diagrams).

Context:
{context}

Question:
{query}
"""
  response = model.invoke(prompt)
  final_results["rag"].append({
        "query_index": i,
        "query": query,
        "expected_answer": test["correct_answer"],
        "rag_response": response.content,
        #"expected_page": test["expected_page"],
        #"retrieved_pages" : retrieved_pages,
        #"Retrieval Accuracy": is_hit,
        #"hits": hits,
        #"contexts": [d.page_content for d in docs]
  })



In [ ]:
import pandas as pd
df_final_rag = pd.DataFrame(final_results['rag'])
df_final_rag

In [33]:
df_final_rag.to_csv('/content/rag_final_results.csv', index=False)

In [ ]:
#first testing round using 30-question set
print("testing baseline model, no context...")
results['baseline'] = []
for i, test in enumerate(dataset):
  query = test["query"]
  prompt = f"""
          You are a discrete math tutor.

          Answer the following question clearly and show all reasoning steps.

          Question: {query}
          """
  response = baseline_model.invoke(prompt)
  results["baseline"].append({
        "query_index": i,
        "query": query,
        "expected_answer": test["correct_answer"],
        "baseline_response": response.content,
  })




In [ ]:
import pandas as pd
df_baseline = pd.DataFrame(results['baseline'])
df_baseline

In [ ]:
df_baseline.to_csv('/content/baseline_results.csv', index=False)

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
print("testing RAG model with context...")
results['rag'] = []
hits = 0
for i, test in enumerate(dataset):
  query = test["query"]
  docs = vectorstore.similarity_search(query, k=4)

  context = "\n\n".join(d.page_content for d in docs)
  retrieved_pages = [int(d.metadata.get("page_label",0)) for d in docs]

  is_hit = test["expected_page"] in retrieved_pages
  if is_hit:
    hits += 1

  prompt = f"""
You are an expert CS173 discrete mathematics tutor.

Answer using only the provided context.

Rules:
- Use precise mathematical language.
- Make sure to use the same notation as the context (e.g. $W_n$)
- Base all definitions and claims strictly on the context.
- If the context is insufficient, explicitly say so.
- Justify all claims clearly and logically.
- Follow proof style consistent with the context when applicable.

Problem-specific guidance:
- For state machine creation: identify required state memory, define transitions, then check determinism and check if your solution can be optimized by merging states.
- Proofs: follow standard discrete math proof structure (direct, contradiction, induction as appropriate).
- For counting and set-theoretic questions, carefully distinguish finite, countably infinite, and uncountable sets.

Formatting:
- Present structured objects (trees, automata, diagrams) in clean readable formats (tables, dictionaries, or ASCII diagrams).

Context:
{context}

Question:
{query}
"""
  response = model.invoke(prompt)
  results["rag"].append({
        "query_index": i,
        "query": query,
        "expected_answer": test["correct_answer"],
        "rag_response": response.content,
        "expected_page": test["expected_page"],
        "retrieved_pages" : retrieved_pages,
        "Retrieval Accuracy": is_hit,
        "hits": hits,
        "contexts": [d.page_content for d in docs]
  })




In [ ]:
import pandas as pd
df_rag = pd.DataFrame(results['rag'])
df_rag

In [ ]:
df_rag.to_csv('rag_results_finalpromptF.csv', index=False)

In [ ]:
!pip install --upgrade ragas

In [ ]:
import sys
from types import ModuleType
from langchain_core.language_models import BaseLanguageModel

mock_chat_models = ModuleType("vertexai")
mock_llms = ModuleType("vertexai")

class DummyVertex(BaseLanguageModel):
    def _generate(self, *args, **kwargs): pass
    def _llm_type(self) -> str: return "vertexai"

mock_chat_models.ChatVertexAI = DummyVertex
mock_llms.VertexAI = DummyVertex

sys.modules['langchain_community.chat_models.vertexai'] = mock_chat_models
sys.modules['langchain_community.llms.vertexai'] = mock_llms

print("Successfully re-applied legacy VertexAI patches!")

In [ ]:
from ragas import evaluate
from datasets import Dataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

In [24]:
ragas_data = {
    "question":     [r["query"]           for r in results["rag"]],
    "answer":       [r["rag_response"]    for r in results["rag"]],
    "contexts":     [r["contexts"]        for r in results["rag"]],
    "ground_truth": [r["expected_answer"] for r in results["rag"]]
}

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from datasets import Dataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", max_tokens=4096))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

ragas_dataset = Dataset.from_dict(ragas_data)

ragas_results = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print(ragas_results)
ragas_results.to_pandas().to_csv("ragas_results.csv", index=False)